This notebook uses the Customers and Transactions tables to form a country-based table for visualization purposes.

Code written by Chanakya Gajjar

In [0]:
#imports
from pyspark.sql.functions import avg, count, round, col, sum
from pyspark.sql.window import Window
import pyspark.sql.functions as F

# In Databricks, these tables were read from the Silver layer:
# spark.table("jarvis_dev.silver.transactions")
# spark.table("jarvis_dev.silver.customers")

# For reproducibility in this repository, we load the raw datasets locally.

#loading the files
transactions = spark.read.csv("transactions.csv", header=True, inferSchema=True)
customers = spark.read.csv("customers.csv", header=True, inferSchema=True)

In [0]:
#creating a table including customer info and country info for each transaction
transactionsWithCountry = transactions.join(customers.select("customer_id", "country"), on = "customer_id", how = "inner")

#grouping the new table by country
transactionsWithCountry = transactionsWithCountry.groupBy("country").agg(
    count("*").alias("Transactions"),
    round(avg("order_value"),2).alias("Average_Order_Amount"),
    round(sum("order_value"),2).alias("Total_Revenue"),
    )


In [0]:

#grouping th customers table by country and retreiving average statistics
countryStats = customers.groupBy("country").agg(
    count("*").alias("Customers"),
    round(avg("age"),2).alias("Average_Customer_Age"),
    round(avg("loyalty_score"),2).alias("Average_Loyalty_Score"),
    round(avg("lifetime_value"),2).alias("Average_Lifetime_Value")
)

countryStats.display()

In [0]:
#joining both tables by the countries and writing it into a gold file
countryStats = countryStats.join(transactionsWithCountry, on = "country", how = "inner")
countryStats.display()
#This line created the gold layer file for this notebook
#countryStats.write.mode("overwrite").saveAsTable("jarvis_dev.gold.gold_country_stats") 


Databricks visualization. Run in Databricks to view.